In [1]:
def main(datasources, start_date, end_date):
    """AI_XOR_01: XGBoost all-base orthogonal residual factor.

    The model is trained only on the fixed 2019-2023 historical interval.
    The platform-injected interval is prediction-only.  Exposure.ret is used
    solely as a strict next-trading-day training label and is never a feature.
    The daily factor is available only after the close of its date.
    """
    import gc
    import numpy as np
    import pandas as pd
    import dai
    from xgboost import XGBRegressor

    prediction_table = datasources["bar1m"]
    historical_bar_table = "bigalpha_2026_stock_bar1m"
    prediction_start = pd.Timestamp(start_date).normalize()
    prediction_end = pd.Timestamp(end_date).normalize()
    if prediction_end < prediction_start:
        raise ValueError("end_date cannot be earlier than start_date")

    # Fixed ex-ante training window.  The min guard preserves strict separation
    # if the platform ever requests a prediction interval before 2024.
    training_start = pd.Timestamp("2019-01-01")
    training_end = min(pd.Timestamp("2023-12-31"), prediction_start - pd.Timedelta(days=1))
    if training_end < pd.Timestamp("2021-01-01"):
        raise ValueError("prediction interval leaves too little historical training data")

    RAW_NUMERIC = [
        "minute_count", "day_open", "day_close", "day_high", "day_low",
        "early_price", "morning_price", "afternoon_price", "late_price",
        "total_volume", "total_amount", "total_deals", "early_volume", "late_volume",
        "valid_book_minutes", "spread_mean", "spread_std", "imbalance_mean",
        "imbalance_std", "early_imbalance", "late_imbalance", "depth_mean",
        "early_depth", "late_depth", "near_depth_share_mean", "near_depth_share_std",
        "book_slope_mean", "book_slope_std",
    ]
    BASE_FEATURES = [
        "ret_open_close", "range_relative", "close_location", "early_late_return",
        "curve_reversal", "log_volume", "log_amount", "log_deals",
        "log_amount_per_deal", "volume_time_skew", "spread_level",
        "spread_variation", "imbalance_level", "imbalance_variation",
        "imbalance_time_shift", "log_visible_depth", "depth_time_shift",
        "near_depth_share", "near_depth_variation", "book_slope_level",
        "book_slope_variation",
    ]
    V6_FEATURES = ['range_relative', 'early_late_return', 'curve_reversal', 'log_volume', 'log_amount', 'log_deals', 'log_amount_per_deal', 'volume_time_skew', 'spread_level', 'imbalance_level', 'log_visible_depth', 'near_depth_share']

    def _normalize_keys(frame):
        result = frame.copy()
        result["date"] = pd.to_datetime(result["date"], errors="coerce").dt.normalize()
        result["instrument"] = result["instrument"].astype(str).str.strip()
        if result["date"].isna().any():
            raise ValueError("date 存在无法解析的值")
        if result["instrument"].eq("").any():
            raise ValueError("instrument 存在空字符串")
        return result

    def _month_windows(start_date, end_date):
        start = pd.Timestamp(start_date).normalize()
        end = pd.Timestamp(end_date).normalize()
        cursor = start.replace(day=1)
        windows = []
        while cursor <= end:
            next_month = cursor + pd.offsets.MonthBegin(1)
            month_start = max(start, cursor)
            month_end = min(end, next_month - pd.Timedelta(seconds=1))
            windows.append((month_start, month_end))
            cursor = next_month
        return windows

    def _safe_divide(numerator, denominator):
        num = np.asarray(numerator, dtype=np.float64)
        den = np.asarray(denominator, dtype=np.float64)
        out = np.full(num.shape, np.nan, dtype=np.float64)
        valid = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > 1e-12)
        out[valid] = num[valid] / den[valid]
        return out

    def _query_stock_pool(start_date, end_date):
        pool = dai.query(
            """
            SELECT date, instrument
            FROM bigalpha_2026_instruments
            """,
            filters={
                "date": [
                    pd.Timestamp(start_date).strftime("%Y-%m-%d 00:00:00"),
                    pd.Timestamp(end_date).strftime("%Y-%m-%d 23:59:59"),
                ]
            },
            compression=True,
        ).df()
        pool = _normalize_keys(pool)
        if pool.duplicated(["date", "instrument"]).any():
            raise ValueError("动态股票池存在重复键")
        return pool

    def _query_daily_month(bar1m_table, month_start, month_end):
        # 一分钟快照只作为可见盘口状态，不解释为真实撤单、主动买卖或队列行为。
        sql = f"""
        WITH minute_base AS (
            SELECT
                date,
                date::DATE::DATETIME AS trading_day,
                instrument,
                STRFTIME(date, '%H:%M') AS hm,
                CAST(open AS DOUBLE) AS px_open,
                CAST(high AS DOUBLE) AS px_high,
                CAST(low AS DOUBLE) AS px_low,
                CAST(close AS DOUBLE) AS px_close,
                CAST(COALESCE(volume, 0) AS DOUBLE) AS minute_volume,
                CAST(COALESCE(amount, 0) AS DOUBLE) AS minute_amount,
                CAST(COALESCE(deal_number, 0) AS DOUBLE) AS minute_deals,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                     AND ask_price1 >= bid_price1
                    THEN (ask_price1 + bid_price1) / 2.0
                    ELSE NULL
                END AS mid_price,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                     AND ask_price1 >= bid_price1
                    THEN (ask_price1 - bid_price1)
                         / ((ask_price1 + bid_price1) / 2.0 + 1e-8)
                    ELSE NULL
                END AS relative_spread,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                     AND ask_price1 >= bid_price1
                    THEN (
                        5.0 * CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + 4.0 * CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                      + 3.0 * CAST(COALESCE(bid_volume3, 0) AS DOUBLE)
                      + 2.0 * CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                      +       CAST(COALESCE(bid_volume5, 0) AS DOUBLE)
                      - 5.0 * CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                      - 4.0 * CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                      - 3.0 * CAST(COALESCE(ask_volume3, 0) AS DOUBLE)
                      - 2.0 * CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                      -       CAST(COALESCE(ask_volume5, 0) AS DOUBLE)
                    ) / (
                        5.0 * CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + 4.0 * CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                      + 3.0 * CAST(COALESCE(bid_volume3, 0) AS DOUBLE)
                      + 2.0 * CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                      +       CAST(COALESCE(bid_volume5, 0) AS DOUBLE)
                      + 5.0 * CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                      + 4.0 * CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                      + 3.0 * CAST(COALESCE(ask_volume3, 0) AS DOUBLE)
                      + 2.0 * CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                      +       CAST(COALESCE(ask_volume5, 0) AS DOUBLE)
                      + 1e-8
                    )
                    ELSE NULL
                END AS weighted_imbalance,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                    THEN (
                        CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume5, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume5, 0) AS DOUBLE)
                    )
                    ELSE NULL
                END AS visible_depth,
                CASE
                    WHEN ask_price1 > 0 AND bid_price1 > 0
                    THEN (
                        CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                    ) / (
                        CAST(COALESCE(bid_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(bid_volume5, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume1, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume2, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume3, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume4, 0) AS DOUBLE)
                      + CAST(COALESCE(ask_volume5, 0) AS DOUBLE)
                      + 1e-8
                    )
                    ELSE NULL
                END AS near_depth_share,
                CASE
                    WHEN ask_price5 > 0 AND ask_price1 > 0
                     AND bid_price1 > 0 AND bid_price5 > 0
                    THEN (
                        (ask_price5 - ask_price1)
                      + (bid_price1 - bid_price5)
                    ) / ((ask_price1 + bid_price1) / 2.0 + 1e-8)
                    ELSE NULL
                END AS book_slope
            FROM {bar1m_table}
        )
        SELECT
            trading_day AS date,
            instrument,
            COUNT(px_close) AS minute_count,
            FIRST(px_open ORDER BY date) AS day_open,
            LAST(px_close ORDER BY date) AS day_close,
            MAX(px_high) AS day_high,
            MIN(px_low) AS day_low,
            AVG(CASE WHEN hm BETWEEN '09:31' AND '10:00' THEN px_close END) AS early_price,
            AVG(CASE WHEN hm BETWEEN '10:01' AND '11:30' THEN px_close END) AS morning_price,
            AVG(CASE WHEN hm BETWEEN '13:01' AND '14:00' THEN px_close END) AS afternoon_price,
            AVG(CASE WHEN hm BETWEEN '14:31' AND '15:00' THEN px_close END) AS late_price,
            SUM(minute_volume) AS total_volume,
            SUM(minute_amount) AS total_amount,
            SUM(minute_deals) AS total_deals,
            SUM(CASE WHEN hm BETWEEN '09:31' AND '10:00' THEN minute_volume ELSE 0 END) AS early_volume,
            SUM(CASE WHEN hm BETWEEN '14:31' AND '15:00' THEN minute_volume ELSE 0 END) AS late_volume,
            COUNT(relative_spread) AS valid_book_minutes,
            AVG(relative_spread) AS spread_mean,
            NANSTD(relative_spread) AS spread_std,
            AVG(weighted_imbalance) AS imbalance_mean,
            NANSTD(weighted_imbalance) AS imbalance_std,
            AVG(CASE WHEN hm BETWEEN '09:31' AND '10:00' THEN weighted_imbalance END) AS early_imbalance,
            AVG(CASE WHEN hm BETWEEN '14:31' AND '15:00' THEN weighted_imbalance END) AS late_imbalance,
            AVG(visible_depth) AS depth_mean,
            AVG(CASE WHEN hm BETWEEN '09:31' AND '10:00' THEN visible_depth END) AS early_depth,
            AVG(CASE WHEN hm BETWEEN '14:31' AND '15:00' THEN visible_depth END) AS late_depth,
            AVG(near_depth_share) AS near_depth_share_mean,
            NANSTD(near_depth_share) AS near_depth_share_std,
            AVG(book_slope) AS book_slope_mean,
            NANSTD(book_slope) AS book_slope_std
        FROM minute_base
        GROUP BY trading_day, instrument
        """
        frame = dai.query(
            sql,
            filters={
                "date": [
                    pd.Timestamp(month_start).strftime("%Y-%m-%d 00:00:00"),
                    pd.Timestamp(month_end).strftime("%Y-%m-%d 23:59:59"),
                ]
            },
            compression=True,
        ).df()
        return _normalize_keys(frame)

    def _derive_base_features(frame):
        result = frame[["date", "instrument"] + RAW_NUMERIC].copy()
        for column in RAW_NUMERIC:
            result[column] = pd.to_numeric(result[column], errors="coerce")
        result[RAW_NUMERIC] = result[RAW_NUMERIC].replace([np.inf, -np.inf], np.nan)

        result["ret_open_close"] = _safe_divide(result["day_close"], result["day_open"]) - 1.0
        result["range_relative"] = _safe_divide(
            result["day_high"] - result["day_low"], result["day_close"]
        )
        result["close_location"] = _safe_divide(
            result["day_close"] - result["day_low"],
            result["day_high"] - result["day_low"],
        ) - 0.5
        result["early_late_return"] = _safe_divide(
            result["late_price"], result["early_price"]
        ) - 1.0
        result["curve_reversal"] = (
            _safe_divide(result["afternoon_price"], result["morning_price"]) - 1.0
        ) - result["ret_open_close"]
        result["log_volume"] = np.log1p(result["total_volume"].clip(lower=0))
        result["log_amount"] = np.log1p(result["total_amount"].clip(lower=0))
        result["log_deals"] = np.log1p(result["total_deals"].clip(lower=0))
        result["log_amount_per_deal"] = np.log1p(
            _safe_divide(result["total_amount"], result["total_deals"] + 1.0)
        )
        result["volume_time_skew"] = _safe_divide(
            result["late_volume"] - result["early_volume"],
            result["total_volume"] + 1.0,
        )
        result["spread_level"] = result["spread_mean"]
        result["spread_variation"] = result["spread_std"]
        result["imbalance_level"] = result["imbalance_mean"]
        result["imbalance_variation"] = result["imbalance_std"]
        result["imbalance_time_shift"] = result["late_imbalance"] - result["early_imbalance"]
        result["log_visible_depth"] = np.log1p(result["depth_mean"].clip(lower=0))
        result["depth_time_shift"] = np.log1p(result["late_depth"].clip(lower=0)) - np.log1p(
            result["early_depth"].clip(lower=0)
        )
        result["near_depth_share"] = result["near_depth_share_mean"]
        result["near_depth_variation"] = result["near_depth_share_std"]
        result["book_slope_level"] = result["book_slope_mean"]
        result["book_slope_variation"] = result["book_slope_std"]

        result = result[["date", "instrument"] + BASE_FEATURES]
        result[BASE_FEATURES] = result[BASE_FEATURES].replace([np.inf, -np.inf], np.nan)
        return result

    def _load_daily_base(bar1m_table, start_date, end_date, label):
        pieces = []
        windows = _month_windows(start_date, end_date)
        for index, (month_start, month_end) in enumerate(windows, start=1):
            print(
                f"{label} 月度聚合 {index:02d}/{len(windows):02d}："
                f"{month_start:%Y-%m-%d} 至 {month_end:%Y-%m-%d}"
            )
            daily = _query_daily_month(bar1m_table, month_start, month_end)
            pool = _query_stock_pool(month_start, month_end)
            merged = pool.merge(
                daily,
                how="left",
                on=["date", "instrument"],
                validate="one_to_one",
            )
            derived = _derive_base_features(merged)
            pieces.append(derived)
            print("  股票日行数：", len(derived))
            del daily, pool, merged, derived
            gc.collect()

        result = pd.concat(pieces, ignore_index=True)
        del pieces
        gc.collect()
        result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
        if result.duplicated(["date", "instrument"]).any():
            raise ValueError(f"{label} 底座存在重复键")
        return result

    def _load_strict_t1(start_date, end_date):
        returns = dai.query(
            """
            SELECT date, instrument, ret
            FROM bigalpha_2026_exposure
            """,
            filters={
                "date": [
                    pd.Timestamp(start_date).strftime("%Y-%m-%d 00:00:00"),
                    pd.Timestamp(end_date).strftime("%Y-%m-%d 23:59:59"),
                ]
            },
            compression=True,
        ).df()
        returns = _normalize_keys(returns)
        returns["ret"] = pd.to_numeric(returns["ret"], errors="coerce").replace(
            [np.inf, -np.inf], np.nan
        )
        if returns.duplicated(["date", "instrument"]).any():
            raise ValueError("收益源存在重复键")

        trading_dates = pd.DatetimeIndex(sorted(returns["date"].unique()))
        if len(trading_dates) < 2:
            raise ValueError("收益源交易日不足，无法构造严格T+1")
        mapping = pd.DataFrame({
            "date": trading_dates[:-1],
            "target_date": trading_dates[1:],
        })
        targets = mapping.merge(
            returns[["date", "instrument", "ret"]].rename(
                columns={"date": "target_date", "ret": "ret_t1"}
            ),
            how="left",
            on="target_date",
            validate="one_to_many",
        )[["date", "instrument", "ret_t1"]]
        del returns, mapping
        gc.collect()
        return targets

    def _attach_target_and_rank(frame, matrix, start_date, end_date):
        targets = _load_strict_t1(start_date, end_date)
        keys = frame[["date", "instrument"]].copy()
        keys["row_id"] = np.arange(len(keys), dtype=np.int64)
        joined = keys.merge(
            targets,
            how="left",
            on=["date", "instrument"],
            validate="one_to_one",
        ).sort_values("row_id")
        if not np.array_equal(joined["row_id"].to_numpy(), np.arange(len(frame))):
            raise ValueError("标签连接后行序发生变化")
        target_rank = joined["ret_t1"].groupby(joined["date"], observed=True).rank(
            method="average", pct=True
        )
        y = ((target_rank.to_numpy(dtype=np.float32) - 0.5) * 2.0).astype(np.float32)
        dates = joined["date"].to_numpy()
        del targets, keys, joined, target_rank
        gc.collect()
        return matrix, y, dates

    def _date_slices(dates):
        dates = np.asarray(dates)
        if len(dates) == 0:
            return []
        boundaries = np.flatnonzero(dates[1:] != dates[:-1]) + 1
        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(dates)]
        return [(int(start), int(end), pd.Timestamp(dates[start])) for start, end in zip(starts, ends)]

    def _residualize_v6(values, controls, dates, min_count=600):
        result = np.full(len(values), np.nan, dtype=np.float32)
        for start_index, end_index, _ in _date_slices(dates):
            y = np.asarray(values[start_index:end_index], dtype=np.float64)
            x = np.asarray(controls[start_index:end_index], dtype=np.float64)
            if x.ndim == 1:
                x = x[:, None]
            valid = np.isfinite(y) & np.all(np.isfinite(x), axis=1)
            if valid.sum() < min_count:
                continue
            xv = x[valid]
            scale = xv.std(axis=0, ddof=1)
            usable = np.isfinite(scale) & (scale > 1e-12)
            xv = xv[:, usable]
            design = np.column_stack([np.ones(valid.sum()), xv])
            gram = design.T @ design
            penalty = 1e-6 * np.eye(gram.shape[0])
            penalty[0, 0] = 0.0
            beta = np.linalg.solve(gram + penalty, design.T @ y[valid])
            block = np.full(end_index - start_index, np.nan, dtype=np.float32)
            block[valid] = (y[valid] - design @ beta).astype(np.float32)
            result[start_index:end_index] = block
        return result

    def _time_decay_weights_v6(dates):
        # 固定约两年半衰期，不由验证结果调参；较近训练样本权重更高。
        normalized = pd.to_datetime(np.asarray(dates)).normalize()
        age_days = np.asarray(
            (normalized.max() - normalized).days,
            dtype=np.float64,
        )
        weight = np.power(0.5, age_days / 730.0)
        return np.clip(weight, 0.15, 1.0).astype(np.float32)

    def rank_fixed_features(frame):
        matrix = np.empty((len(frame), len(V6_FEATURES)), dtype=np.float32)
        grouped = frame.groupby("date", observed=True, sort=False)
        stock_count = grouped["instrument"].size()
        for column_index, feature in enumerate(V6_FEATURES):
            valid_count = grouped[feature].count()
            missing_rate = 1.0 - valid_count / stock_count
            unique_count = grouped[feature].nunique(dropna=True)
            if valid_count.min() < 600 or missing_rate.max() > 0.40 or unique_count.min() < 100:
                raise ValueError(f"base feature quality gate failed: {feature}")
            neutral = frame[feature].fillna(grouped[feature].transform("median"))
            ranked = neutral.groupby(frame["date"], observed=True).rank(method="average", pct=True)
            matrix[:, column_index] = (ranked.to_numpy(dtype=np.float32) - 0.5) * 2.0
        return matrix

    def existing_matrix(matrix):
        # Exact reconstruction of the four frozen symbolic factors used by v6.
        existing_lsr = -np.where(matrix[:, 10] >= 0.0, matrix[:, 0], matrix[:, 1])
        existing_vpcr = matrix[:, 7] * matrix[:, 2]
        existing_vwa = -(matrix[:, 0] * np.maximum(matrix[:, 3], 0.0))
        existing_cvsr = matrix[:, 2] * np.maximum(matrix[:, 7], 0.0)
        # Same alphabetical order as the frozen research implementation.
        return np.column_stack([
            existing_cvsr, existing_lsr, existing_vpcr, existing_vwa,
        ]).astype(np.float32, copy=False)

    model_params = {'max_depth': 3, 'n_estimators': 180, 'learning_rate': 0.03, 'min_child_weight': 80, 'subsample': 0.8, 'colsample_bytree': 0.75, 'reg_alpha': 2.0, 'reg_lambda': 12.0, 'random_state': 20260823}
    model = XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=2,
        max_bin=128,
        verbosity=0,
        **model_params,
    )

    train_base = _load_daily_base(
        historical_bar_table, training_start, training_end, "fixed training"
    )
    train_matrix = rank_fixed_features(train_base)
    train_matrix, train_y, train_dates = _attach_target_and_rank(
        train_base, train_matrix, training_start, training_end
    )
    train_existing_matrix = existing_matrix(train_matrix)
    train_target = train_y
    train_valid = np.isfinite(train_target) & np.all(np.isfinite(train_matrix), axis=1)
    if train_valid.sum() < 200000:
        raise ValueError("valid training sample count below 200000")
    train_weights = _time_decay_weights_v6(train_dates)[train_valid]
    model.fit(
        train_matrix[train_valid], train_target[train_valid], sample_weight=train_weights
    )
    del train_base, train_y, train_target, train_existing_matrix, train_valid, train_weights
    gc.collect()

    prediction_base = _load_daily_base(
        prediction_table, prediction_start, prediction_end, "prediction"
    )
    prediction_matrix = rank_fixed_features(prediction_base)
    prediction_dates = prediction_base["date"].to_numpy()
    prediction_existing_matrix = existing_matrix(prediction_matrix)
    raw_prediction = model.predict(prediction_matrix).astype(np.float32)
    prediction_controls = np.column_stack([prediction_matrix, prediction_existing_matrix]).astype(np.float32, copy=False)
    factor = _residualize_v6(
        raw_prediction, prediction_controls, prediction_dates, min_count=600
    )

    result = prediction_base[["date", "instrument"]].copy()
    result["factor"] = pd.Series(factor, index=result.index, dtype="float64").replace(
        [np.inf, -np.inf], np.nan
    )
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("duplicate factor output key")
    quality = result.groupby("date", observed=True)["factor"].agg(
        valid_count="count", unique_count="nunique", daily_std="std"
    )
    daily_count = result.groupby("date", observed=True).size()
    if (1.0 - quality["valid_count"] / daily_count).max() > 0.40:
        raise ValueError("daily factor missing rate exceeds 40%")
    if quality["valid_count"].min() < 600 or quality["unique_count"].min() < 200:
        raise ValueError("factor cross-sectional coverage is insufficient")
    if quality["daily_std"].fillna(0).min() <= 0:
        raise ValueError("factor has a constant daily cross-section")
    return result[["date", "instrument", "factor"]]
